# Ampliación histórica: evidencia de la carga

La base `economia_rd` añade series nacionales, educación, compras y encuestas complementarias. Este cuaderno **revisa archivos de evidencia ya generados**: no consulta PostgreSQL ni descarga datos. La verificación en vivo está en `verify-expansion.R`; la comparación CSV/SQL, en `check-roundtrip.R`. No contiene contraseñas.

## Método y supuestos
Se preservan valores originales y se tipan fechas, identificadores y medidas. Las vistas añaden unidades, etiquetas, períodos y banderas de calidad. Las filas de tablas relacionadas no son observaciones independientes. No se supone un panel de las mismas personas entre encuestas. Las ausencias y años incompletos no se rellenan. Ejecutar desde la raíz del repositorio o desde esta carpeta.

In [1]:
from pathlib import Path
import csv, json
base = Path.cwd()
if not (base / 'qa/validation-summary.json').exists():
    base = base / 'research/mercado-laboral-dominicano/postgresql-recovery/expansion'
qa = base / 'qa'
def read_json(name):
    return json.loads((qa / name).read_text(encoding='utf-8-sig'))
def read_csv(name):
    with (qa / name).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
summary = read_json('validation-summary.json')
print('Base:', summary['database'])
print('Validación técnica:', summary['validated_at'])
print('Estado:', summary['status'])
print('Tamaño GiB:', round(summary['database_bytes'] / 1024**3, 2))

Base: economia_rd
Validación técnica: 2026-08-30T11:19:46-0400
Estado: validada_con_advertencias
Tamaño GiB: 5.41


## Datos y procedencia
Los manifiestos SHA256 identifican los archivos nacionales, doce paquetes anuales de DGCP y 150 lotes de WDI. El catálogo descargado es más amplio que las hojas interpretadas y cargadas en SQL. La comprobación de archivos también incluye los dos diccionarios WDI y el histórico de dólar.

In [2]:
tables = read_csv('table-inventory.csv')
assert len(tables) == summary['tables'] == 45
assert len({r['tabla'] for r in tables}) == len(tables)
assert sum(int(r['filas']) for r in tables) == summary['rows'] == 17186057
snapshot = read_json('source-snapshot-check.json')
assert snapshot['status'] == 'ok'
assert snapshot['extracted_csv_files'] == 156
print('Tablas inventariadas:', len(tables))
print('Filas físicas inventariadas:', summary['rows'])
print('Archivos descargados verificados:', snapshot['downloaded_files'])
print('CSV extraídos verificados:', snapshot['extracted_csv_files'])

Tablas inventariadas: 45
Filas físicas inventariadas: 17186057
Archivos descargados verificados: 437
CSV extraídos verificados: 156


## Resultados de los controles
Los ceros siguientes prueban las condiciones consultadas, no la exactitud de cada respuesta de la fuente. Se concilian meses/componentes, claves, enlaces y unidades; las advertencias se conservan explícitamente.

In [3]:
checks = read_csv('database-checks.csv')
assert len(checks) == summary['checks'] == 24
assert summary['errors'] == 0
assert all(r['estado'] != 'error' for r in checks)
assert sum(r['estado'] == 'advertencia' for r in checks) == summary['warnings'] == 9
for r in checks:
    if r['estado'] == 'advertencia':
        print(r['control'], ':', r['observado'], '—', r['nota'])

matricula_codigo_periodo_fuente : 140 — Código original con un dígito de más: normalizado explícitamente a 2021-2022 en la vista.
pruebas_componentes_sexo : 185 — Desacuerdo presente en la fuente. No recomponer el total a partir de componentes sin revisar.
pruebas_componentes_resultado : 638 — No imputar el componente para forzar coincidencia.
pruebas_duplicados_exactos_fuente : 2765 — Duplicados idénticos colapsados solamente en la vista analítica; originales intactos.
pruebas_clave_repetida : 494 — Claves con contenidos distintos después de quitar duplicados exactos; se conservan y señalan con clave_ambigua.
fiscal_unidad_pendiente : 948 — Se conserva el original; estas celdas no aparecen en presupuesto_mensual.
fiscal_filas_no_conciliadas : 45 — Suma mensual y total publicado difieren. Se excluye la fila de la vista predeterminada.
dgcp_periodo_contractual_invertido : 33 — Anomalía de fechas publicada por la fuente; no interpretarla como duración válida.
enhogar_elegidos_sin_persona

In [4]:
roundtrip = read_csv('roundtrip-dgcp.csv')
assert len(roundtrip) == 26
assert all(r['estado'] == 'ok' for r in roundtrip)
assert {int(r['anio']) for r in roundtrip} == {2015, 2026}
examples = read_csv('example-queries.csv')
assert len(examples) == 19 and all(r['estado'] == 'ok' for r in examples)
assert all(int(r['filas']) > 0 for r in examples[:-1])
assert int(examples[-1]['filas']) == 0
print('Comparaciones CSV/SQL de todas las columnas de filas muestreadas:', len(roundtrip))
print('Consultas SQL de ejemplo ejecutadas:', len(examples))
print('Última consulta: sin huecos mensuales en el índice general del IPC.')

Comparaciones CSV/SQL de todas las columnas de filas muestreadas: 26
Consultas SQL de ejemplo ejecutadas: 19
Última consulta: sin huecos mensuales en el índice general del IPC.


In [5]:
wdi = read_json('wdi.json')[0]
assert wdi['observaciones'] == 1083106
assert wdi['indicadores_rd'] == 1363
assert wdi['paises_y_agregados'] == 25
assert (wdi['desde'], wdi['hasta']) == (1960, 2025)
print('WDI:', wdi)
for r in summary['coverage']:
    print(r['dataset'], r['desde'], 'a', r['hasta'], r['frecuencia'])

WDI: {'observaciones': 1083106, 'paises_y_agregados': 25, 'indicadores': 1497, 'desde': 1960, 'hasta': 2025, 'observaciones_rd': 47009, 'indicadores_rd': 1363, 'observaciones_educacion': 79000}
canasta_quintiles 2018-01-01 a 2026-07-01 mensual
canasta_regiones 2020-10-01 a 2026-07-01 mensual
credito_destino 1996-01-01 a 2026-07-01 mensual
depositos_provincias 2014-01-01 a 2026-04-01 trimestral
imae_2018 2007-01-01 a 2026-07-01 mensual
ipc_grupos 1999-01-01 a 2026-07-01 mensual
ipc_nacional 1984-01-01 a 2026-07-01 mensual
ipc_quintiles 2020-10-01 a 2026-07-01 mensual
ipc_regiones 2011-01-01 a 2026-07-01 mensual
ipc_subyacente 2000-01-01 a 2026-07-01 mensual
ipc_subyacente_cierre_anual 2000-12-01 a 2025-12-01 anual
pib_origen_2018_acumulado 2007-01-01 a 2026-01-01 trimestral_acumulada
pib_origen_2018_trimestral 2007-01-01 a 2026-01-01 trimestral
politica_monetaria 2004-01-01 a 2026-08-01 mensual
prestamos_provincias 2014-01-01 a 2026-04-01 trimestral
remesas 2010-01-01 a 2026-07-01 mensu

In [6]:
details = read_json('analytical-summary.json')
pruebas = details['pruebas'][0]
assert pruebas['registros'] == 54821
assert pruebas['duplicados_exactos_excluidos'] == 2765
assert pruebas['registros_ambiguos'] == 988
central = read_json('central-registration.json')
assert len(central['views']) == details['views'] == 35
assert len(central['read_checks']) == 5
print('Vistas disponibles en', central['database'] + '.' + central['schema'], ':', len(central['views']))
print('Pruebas nacionales:', pruebas)
print('Educación internacional:', details['wdi_education'][0])

Vistas disponibles en censos_linea_tiempo.fdw_economia : 35
Pruebas nacionales: {'registros': 54821, 'registros_ambiguos': 988, 'duplicados_exactos_excluidos': 2765}
Educación internacional: {'observaciones': 79000, 'indicadores': 156, 'observaciones_rd': 2969}


## Límites y siguiente uso
En MINERD se colapsaron exclusivamente repeticiones exactas; las claves con contenidos distintos siguen marcadas. Los resultados de pruebas son participaciones, no alumnos únicos, y no forman una serie homogénea sin revisar convocatorias/cobertura. En presupuesto se excluyen unidades pendientes y filas no conciliadas de la vista predeterminada. Contrato no equivale a pago; TSS cuenta empleos cotizantes; cada encuesta conserva su factor y universo. El catálogo WDI incluye países y agregados y sus series tienen coberturas distintas.

Consulte [README.md](README.md) y [example-queries.sql](example-queries.sql) antes de calcular resultados. Para renovar esta evidencia, ejecutar los verificadores en vivo, luego `finish-catalog.R`, `register-central.R`, `restore-snapshot.ps1 -VerifyOnly` y reconstruir este cuaderno. No se certifica aquí la comparabilidad sustantiva de los 1.497 indicadores ni cada celda de los originales.